In [ ]:
# Imports
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from principal_agent_mdp import PrincipalAgentMDP
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
from principal_dq import PrincipalDQ
from agent_dq import AgentDQ
from deepq_network import QNetwork
from deepq_network import ReplayBuffer
import time 

            


# update model
def parameter_update(source_model, target_model, tau):
    for target_param, source_param in zip(target_model.parameters(), source_model.parameters()):
        target_param.data.copy_(tau * source_param.data + (1.0 - tau)*target_param.data)

# one hot encoding 
def to_one_hot(s, n_states):
    v = torch.zeros(n_states, dtype=torch.float32).to(device)
    v[s] = 1.0
    return v
# Initialization of Algorithm 2 

# env
EPSILON_START = 1.0
EPSILON_END   = 0.0
GAMMA = 1.0
mdp = PrincipalAgentMDP(gamma=GAMMA)
# principal and agent 
r_p = np.zeros((mdp.n_states, mdp.n_outcomes))
for s in range(mdp.n_states):
    r_p[s, 0] = 14/9
    r_p[s, 1] = 0.0
principal = PrincipalDQ(mdp, r_p, epsilon=EPSILON_START)
agent = AgentDQ(mdp, epsilon=EPSILON_START)

# principal's Q-network theta
q_theta = QNetwork(n_states=mdp.n_states, n_actions = mdp.n_actions, hidden_dim=256).to(device) # online Network 
q_theta_prime = QNetwork(n_states=mdp.n_states, n_actions = mdp.n_actions, hidden_dim=256).to(device) # target Network 

# agent's Q-network phi
Q_phi_bar = QNetwork(n_states=mdp.n_states, n_actions = mdp.n_actions, hidden_dim=256).to(device) # online Network 
Q_phi_bar_prime = QNetwork(n_states=mdp.n_states, n_actions = mdp.n_actions, hidden_dim=256).to(device) # target Network 

# optimizer
q_theta_optimizer = torch.optim.Adam(params=q_theta.parameters(), lr=0.001)
Q_phi_bar_optimizer = torch.optim.Adam(params=Q_phi_bar.parameters(), lr=0.001)

q_theta_prime.load_state_dict(q_theta.state_dict())
Q_phi_bar_prime.load_state_dict(Q_phi_bar.state_dict())

# train each network for 20000 iterations 
num_iterations = 20000
# num environment interactions per iteration
num_interactions = 8 
# starting with explortion rate epsilon = 1, and linearly anneal to 0
# mini-batch size 
BATCH_SIZE = 128
WARMUP = 1000
# learning rate scheduler 
# learning rate init with 0.001 annealed exponentially to 0.0001
q_theta_scheduler   = torch.optim.lr_scheduler.ExponentialLR(
    q_theta_optimizer, gamma=(0.0001/0.001)**(1/num_iterations))
Q_phi_bar_scheduler = torch.optim.lr_scheduler.ExponentialLR(
    Q_phi_bar_optimizer, gamma=(0.0001/0.001)**(1/num_iterations))
# buffer 
transition_buffer = ReplayBuffer(20000)
# rho: s -> tuple (b(L), b(R)) — matches Principal output format
rho = {s: (0.0, 0.0) for s in range(mdp.n_states)}

principal_rewards = []
agent_rewards     = []
loss_theta_log = []
loss_phi_log   = []
# start state
s = mdp.s0
# track time 
start_time = time.time()
for num in tqdm(range(num_iterations)):
    epsilon = EPSILON_START - (EPSILON_START - EPSILON_END) * (num / num_iterations)
    principal.epsilon = epsilon
    agent.epsilon = epsilon
    for i in range(num_interactions):
        # select action a_p with q_theta via epsilon greedy 
        a_p = principal.induce_action(s, q_theta)
        # sample outcome, transition the MDP to next state
        o = mdp.sample_outcome(s, a_p)
        s_next = mdp.T(s, o)
        # get agents reward 
        r_a = mdp.R_agent(s,a_p,rho[s],o)
        # get principals reward
        rwrd_p = principal.r_p[s,o]
        done   = int(mdp.is_terminal(s_next))
        # add transition to buffer 
        transition_buffer.append((s,a_p,r_a,rwrd_p, o, done, s_next))
        principal_rewards.append(rwrd_p)
        agent_rewards.append(r_a)
        if done == 1: 
            # reset mdp, i.e reset to initial state 
            s = mdp.s0
        else: 
            s = s_next 

    if len(transition_buffer) < WARMUP:
        continue     
    # sample mini-batch from transitions
    states, actions, r_agent, r_principal, outcomes, dones, next_states = transition_buffer.sample(BATCH_SIZE)
    
    states_oh      = F.one_hot(states.long(), num_classes=mdp.n_states).float().to(device)
    next_states_oh = F.one_hot(next_states.long(), num_classes=mdp.n_states).float().to(device)

    # Estimate target variables to update theata and phi
    # select next action for Q-learning updates argmax_{a_prime}q_theta(next_states)
    a_p_prime = q_theta(next_states_oh).detach().argmax(dim=1)
    # find optimal contract b_star for each Batch 
    b_star = []
    b_star_prime = []
    for idx in range(BATCH_SIZE):
        s_i      = int(states[idx].item())
        s_i_next = int(next_states[idx].item())
        a_i      = int(actions[idx].item())
        a_i_p    = int(a_p_prime[idx].item())
        Q_bar_s      = Q_phi_bar(to_one_hot(s_i, mdp.n_states)).detach().cpu().numpy()
        Q_bar_s_next = Q_phi_bar(to_one_hot(s_i_next, mdp.n_states)).detach().cpu().numpy()
        b_star.append(principal.find_best_contract(s_i, a_i, Q_bar_s))
        b_star_prime.append(principal.find_best_contract(s_i_next, a_i_p, Q_bar_s_next))


    b_star_o = torch.tensor(
        [b_star[i][int(outcomes[i].item())] for i in range(BATCH_SIZE)],
        dtype=torch.float32).to(device)
    q_target_vals = q_target_vals    = q_theta_prime(next_states_oh)\
    .gather(1, a_p_prime.view(-1,1)).squeeze().detach()
    y_p = r_principal - b_star_o + (1 - dones.float()) * q_target_vals

    E_b_prime = torch.tensor(
    [agent._expected_payment(
        int(next_states[i].item()),
        int(a_p_prime[i].item()),
        b_star_prime[i])
     for i in range(BATCH_SIZE)],
    dtype=torch.float32).to(device)

    Q_phi_prime_vals = Q_phi_bar_prime(next_states_oh)\
    .gather(1, a_p_prime.view(-1,1)).squeeze().detach()
    y_a = r_agent + (1 - dones.float()) * (E_b_prime + Q_phi_prime_vals)


    # L(theta) = sum_mb (q_theta(s, a_p) - y_p(s, a_p))^2
    q_pred_theta = q_pred_theta = q_theta(states_oh).gather(1, actions.view(-1,1)).squeeze()
    loss_theta = F.mse_loss(q_pred_theta, y_p)
    q_theta_optimizer.zero_grad()
    loss_theta.backward()
    loss_theta_log.append(loss_theta.item())
    q_theta_optimizer.step()
    q_theta_scheduler.step()

    #L(phi) = sum_mb (Q_phi_bar(s, a_p) - y_a(s, a_p))^2
    q_pred_phi = q_pred_phi   = Q_phi_bar(states_oh).gather(1, actions.view(-1,1)).squeeze()
    loss_phi = F.mse_loss(q_pred_phi, y_a)
    Q_phi_bar_optimizer.zero_grad()
    loss_phi.backward()
    loss_phi_log.append(loss_phi.item())
    Q_phi_bar_optimizer.step()
    Q_phi_bar_scheduler.step()

    # Target network update every 100 iterations — hard copy (Paper S.35)
    if num % 100 == 0:
        parameter_update(q_theta, q_theta_prime, tau=1.0)
        parameter_update(Q_phi_bar, Q_phi_bar_prime, tau=1.0)
            
elapsed = time.time() - start_time
print(f"Training time: {elapsed:.1f}s ({elapsed/60:.1f} min)")

In [ ]:
def running_mean(x, N):
    cumsum = np.cumsum(np.insert(x, 0, 0))
    return (cumsum[N:] - cumsum[:-N]) / float(N)

plt.figure(figsize=(12,9))
plt.plot(running_mean(principal_rewards, 50), label="Principal reward")
plt.plot(running_mean(agent_rewards, 50), label="Agent reward")
plt.title("DQN — Principal and Agent rewards")
plt.xlabel("Interaction")
plt.legend()
plt.grid()
plt.show()